# 🔥 Semantic Cache Test — Erion Ember + Qwen3.5

> **Mục tiêu:** Test semantic cache service [Erion Ember](https://github.com/EricNguyen1206/erion-ember) kết hợp với LLM Qwen3.5 chạy local qua `llama_cpp`.

---

## 📐 Kiến trúc

```
Notebook (Python)
     │
     ├─► POST /v1/cache/get ──► HIT  ──► Trả response ngay (ms)
     │        (Erion Ember)
     │
     └─► MISS ──► llama_cpp (Qwen3.5) ──► POST /v1/cache/set
```

## 🗺️ Các bước
| Cell | Nội dung |
|------|----------|
| 1 | Cài đặt dependencies |
| 2 | Cấu hình & kết nối Erion Ember |
| 3 | Load Qwen3.5 model |
| 4 | Định nghĩa cache helpers |
| 5 | Test semantic similarity |
| 6 | Benchmark tốc độ |
| 7 | Xem thống kê cache |

---
## 📦 Cell 1 — Cài đặt dependencies

> ⚠️ **Trước khi chạy cell này:** Hãy đảm bảo Erion Ember đang chạy ở local.
> ```bash
> # Terminal 1 — Chạy Erion Ember
> git clone https://github.com/EricNguyen1206/erion-ember
> cd erion-ember
> docker-compose up -d
> ```

In [ ]:
# Cài đặt các thư viện cần thiết
%pip install llama-cpp-python requests ipywidgets

print("✅ Dependencies installed!")

---
## ⚙️ Cell 2 — Cấu hình & kiểm tra kết nối Erion Ember

In [ ]:
import requests
import json
import time
from IPython.display import display, HTML

# ─── CẤU HÌNH ────────────────────────────────────────────────
CACHE_HOST          = "http://localhost:8080"  # Đổi nếu dùng port khác
SIMILARITY_THRESHOLD = 0.80                    # 0.0 – 1.0 (thấp hơn = nhạy hơn)
DEFAULT_TTL          = 3600                    # Giây

# ─── KIỂM TRA KẾT NỐI ────────────────────────────────────────
def check_server():
    try:
        r = requests.get(f"{CACHE_HOST}/health", timeout=3)
        if r.status_code == 200:
            display(HTML('<p>✅ <b>Erion Ember</b> đang chạy tại <code>{}</code></p>'.format(CACHE_HOST)))
            return True
        else:
            display(HTML(f'<p>⚠️ Server phản hồi status <b>{r.status_code}</b></p>'))
            return False
    except Exception as e:
        display(HTML(
            '<p>❌ <b>Không kết nối được Erion Ember!</b></p>'
            '<pre>Lỗi: {}\n\nHãy chạy:\n  docker-compose up -d\ntrong thư mục erion-ember</pre>'.format(e)
        ))
        return False

SERVER_OK = check_server()
print(f"\nCấu hình:")
print(f"  HOST               : {CACHE_HOST}")
print(f"  SIMILARITY_THRESHOLD: {SIMILARITY_THRESHOLD}")
print(f"  DEFAULT_TTL        : {DEFAULT_TTL}s")

---
## 🤖 Cell 3 — Load Qwen3.5 model

> Model sẽ được tự động download từ HuggingFace Hub (~800MB). Chỉ cần tải 1 lần.

In [ ]:
from llama_cpp import Llama

print("⏳ Đang tải model Qwen3.5-0.8B...")
t_start = time.perf_counter()

llm = Llama.from_pretrained(
    repo_id="unsloth/Qwen3.5-0.8B-GGUF",
    filename="Qwen3.5-0.8B-BF16.gguf",
    n_ctx=2048,     # context window
    n_threads=4,    # số CPU thread
    verbose=False,
)

t_load = time.perf_counter() - t_start
print(f"✅ Model loaded! ({t_load:.1f}s)")

---
## 🛠️ Cell 4 — Định nghĩa Cache Helpers & Cached Inference

In [ ]:
# ── Cache GET ──────────────────────────────────────────────────
def cache_get(prompt: str, threshold: float = SIMILARITY_THRESHOLD) -> str | None:
    """Tìm cache theo semantic similarity. Trả None nếu MISS."""
    try:
        res = requests.post(
            f"{CACHE_HOST}/v1/cache/get",
            json={"prompt": prompt, "similarity_threshold": threshold},
            timeout=5,
        )
        if res.status_code == 200:
            return res.json().get("response")
    except requests.RequestException as e:
        print(f"  ⚠️  Cache GET error: {e}")
    return None


# ── Cache SET ──────────────────────────────────────────────────
def cache_set(prompt: str, response: str, ttl: int = DEFAULT_TTL) -> bool:
    """Lưu prompt/response vào cache."""
    try:
        res = requests.post(
            f"{CACHE_HOST}/v1/cache/set",
            json={"prompt": prompt, "response": response, "ttl": ttl},
            timeout=5,
        )
        return res.status_code == 200
    except requests.RequestException as e:
        print(f"  ⚠️  Cache SET error: {e}")
    return False


# ── LLM Inference ─────────────────────────────────────────────
def llm_infer(prompt: str, max_tokens: int = 256) -> str:
    """Gọi Qwen3.5 để sinh câu trả lời."""
    output = llm(
        prompt,
        max_tokens=max_tokens,
        stop=["<|im_end|>", "\n\n\n"],
        echo=False,
    )
    return output["choices"][0]["text"].strip()


# ── Cached Ask (luồng chính) ───────────────────────────────────
def ask(prompt: str, threshold: float = SIMILARITY_THRESHOLD, verbose: bool = True):
    """
    Hỏi với semantic cache:
      HIT  → trả response từ cache (nhanh)
      MISS → gọi LLM → lưu cache
    Trả về dict {response, hit, latency_ms}
    """
    t0 = time.perf_counter()

    cached = cache_get(prompt, threshold)
    if cached is not None:
        ms = (time.perf_counter() - t0) * 1000
        if verbose:
            print(f"  🟢 CACHE HIT  — {ms:.1f} ms")
        return {"response": cached, "hit": True, "latency_ms": ms}

    if verbose:
        print("  🔴 CACHE MISS — gọi LLM...")
    t1 = time.perf_counter()
    response = llm_infer(prompt)
    llm_ms = (time.perf_counter() - t1) * 1000

    ok = cache_set(prompt, response)
    total_ms = (time.perf_counter() - t0) * 1000

    if verbose:
        print(f"  ⏱  LLM: {llm_ms:.0f} ms | Total: {total_ms:.0f} ms | Cached: {'✅' if ok else '❌'}")
    return {"response": response, "hit": False, "latency_ms": total_ms}


print("✅ Helpers đã sẵn sàng!")

---
## 🧪 Cell 5 — Test Semantic Similarity

Chạy nhiều câu hỏi với nghĩa tương tự để kiểm tra cache có nhận ra không.

In [ ]:
# ── Các test case ─────────────────────────────────────────────
# Format: (nhãn, prompt)
TEST_CASES = [
    # Nhóm 1 — Machine Learning
    ("[1] MISS — original",      "What is machine learning?"),
    ("[1] HIT?  — paraphrase",   "Can you explain machine learning to me?"),
    ("[1] HIT?  — rephrase",     "What does machine learning mean?"),
    ("[1] HIT?  — short",        "Define machine learning"),

    # Nhóm 2 — Photosynthesis
    ("[2] MISS — original",      "How does photosynthesis work?"),
    ("[2] HIT?  — similar",      "Explain the process of photosynthesis"),

    # Nhóm 3 — Khác hoàn toàn (phải MISS)
    ("[3] MISS — unrelated",     "What is the capital of France?"),
    ("[3] MISS — unrelated",     "How do I make pasta carbonara?"),
]

# ── Chạy test ─────────────────────────────────────────────────
print("=" * 65)
print("  🧪 SEMANTIC CACHE TEST")
print("=" * 65)

results = []
for label, prompt in TEST_CASES:
    print(f"\n{label}")
    print(f"  Prompt : \"{prompt}\"")
    r = ask(prompt)
    preview = r["response"][:90] + ("..." if len(r["response"]) > 90 else "")
    print(f"  Answer : {preview}")
    results.append({"label": label, "prompt": prompt, **r})

# ── Tóm tắt ────────────────────────────────────────────────────
hits  = sum(1 for r in results if r["hit"])
total = len(results)
print("\n" + "=" * 65)
print(f"  📊 Kết quả: {hits} HIT / {total} tổng  |  Hit rate: {hits/total*100:.0f}%")
print("=" * 65)

---
## 🎛️ Cell 5b — Thử nghĩa threshold khác nhau

Dùng slider để điều chỉnh `similarity_threshold` và xem ảnh hưởng.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

prompt_a = "What is artificial intelligence?"
prompt_b = "Can you explain AI to me?"

# Seed cache với prompt_a
print(f"Seeding cache với: \"{prompt_a}\"")
ask(prompt_a, verbose=True)

slider = widgets.FloatSlider(
    value=0.80, min=0.50, max=1.00, step=0.05,
    description="Threshold:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)
output = widgets.Output()

def on_change(change):
    with output:
        clear_output(wait=True)
        th = change["new"]
        print(f"\n🔍 Thử prompt: \"{prompt_b}\"  |  threshold={th:.2f}")
        r = ask(prompt_b, threshold=th)
        status = "🟢 HIT" if r["hit"] else "🔴 MISS"
        print(f"  Kết quả: {status}  ({r['latency_ms']:.1f} ms)")

slider.observe(on_change, names="value")
display(widgets.VBox([slider, output]))

---
## ⚡ Cell 6 — Benchmark: Cache vs LLM

In [ ]:
BENCH_PROMPT = "What is deep learning and how is it different from machine learning?"
N_RUNS       = 6  # Số lần lặp

print("=" * 65)
print(f"  ⚡ BENCHMARK  (N={N_RUNS})")
print(f"  Prompt: \"{BENCH_PROMPT[:55]}...\"")
print("=" * 65)

times_hit, times_miss = [], []

for i in range(N_RUNS):
    t0 = time.perf_counter()
    cached = cache_get(BENCH_PROMPT)

    if cached is not None:
        ms = (time.perf_counter() - t0) * 1000
        times_hit.append(ms)
        print(f"  Run {i+1:02d}: 🟢 HIT   {ms:7.2f} ms")
    else:
        t1 = time.perf_counter()
        resp = llm_infer(BENCH_PROMPT)
        ms = (time.perf_counter() - t1) * 1000
        cache_set(BENCH_PROMPT, resp)
        times_miss.append(ms)
        print(f"  Run {i+1:02d}: 🔴 MISS  {ms:7.2f} ms  (LLM)")

print("\n" + "-" * 65)
if times_miss:
    avg_miss = sum(times_miss) / len(times_miss)
    print(f"  LLM avg (MISS) : {avg_miss:,.0f} ms")
if times_hit:
    avg_hit = sum(times_hit) / len(times_hit)
    print(f"  Cache avg (HIT): {avg_hit:.1f} ms")
if times_hit and times_miss:
    speedup = avg_miss / avg_hit
    print(f"  🚀 Speedup     : {speedup:.0f}x faster with cache")
print("=" * 65)

---
## 📈 Cell 6b — Biểu đồ latency

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Thu thập dữ liệu benchmark cho biểu đồ
BENCH_PROMPTS = [
    "What is neural network?",
    "Explain supervised learning",
    "What is gradient descent?",
    "How does backpropagation work?",
]

miss_times, hit_times = [], []
labels = []

print("Đang thu thập dữ liệu cho biểu đồ...")
for p in BENCH_PROMPTS:
    # Lần 1: MISS
    t0 = time.perf_counter()
    r  = llm_infer(p)
    miss_ms = (time.perf_counter() - t0) * 1000
    cache_set(p, r)

    # Lần 2: HIT
    t0 = time.perf_counter()
    cache_get(p)
    hit_ms = (time.perf_counter() - t0) * 1000

    miss_times.append(miss_ms)
    hit_times.append(hit_ms)
    short = p[:22] + "..."
    labels.append(short)
    print(f"  {short:<26} MISS={miss_ms:.0f}ms  HIT={hit_ms:.1f}ms")

# Plot
x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, miss_times, w, label="LLM (Cache MISS)", color="#ef4444", alpha=0.85)
bars2 = ax.bar(x + w/2, hit_times,  w, label="Cache HIT",         color="#22c55e", alpha=0.85)

ax.set_yscale("log")
ax.set_ylabel("Latency (ms) — log scale", fontsize=12)
ax.set_title("⚡ Semantic Cache: LLM vs Cache Latency", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.legend(fontsize=11)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f"{bar.get_height():.0f}ms", ha="center", va="bottom", fontsize=8, color="#991b1b")
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f"{bar.get_height():.1f}ms", ha="center", va="bottom", fontsize=8, color="#166534")

plt.tight_layout()
plt.savefig("latency_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n💾 Đã lưu biểu đồ: latency_chart.png")

---
## 📊 Cell 7 — Thống kê cache server

In [ ]:
try:
    res = requests.get(f"{CACHE_HOST}/v1/stats", timeout=5)
    stats = res.json()

    print("📊 ERION EMBER — CACHE STATS")
    print("=" * 40)
    for k, v in stats.items():
        print(f"  {k:<25}: {v}")

    # Tính hit rate nếu có đủ field
    hits   = stats.get("hits", 0)
    misses = stats.get("misses", 0)
    total  = hits + misses
    if total > 0:
        print(f"\n  Hit rate: {hits}/{total} = {hits/total*100:.1f}%")

except Exception as e:
    print(f"⚠️  Không lấy được stats: {e}")

---
## 🗑️ Cell 8 — Dọn dẹp (xóa toàn bộ cache)

In [ ]:
# Chỉ chạy khi muốn xóa cache để test lại từ đầu
prompts_to_delete = [
    "What is machine learning?",
    "How does photosynthesis work?",
    "What is the capital of France?",
    "How do I make pasta carbonara?",
    "What is artificial intelligence?",
    BENCH_PROMPT,
] + BENCH_PROMPTS

deleted = 0
for p in prompts_to_delete:
    try:
        r = requests.post(
            f"{CACHE_HOST}/v1/cache/delete",
            json={"prompt": p},
            timeout=5,
        )
        if r.status_code == 200:
            deleted += 1
    except Exception:
        pass

print(f"🗑️  Đã xóa {deleted}/{len(prompts_to_delete)} cache entries")
print("Cache đã được reset. Bạn có thể chạy lại Cell 5 để test từ đầu.")